# GAG-pathway selection-test multiple-testing correction

**Reviewer #4** (comment 6) noted the lineage-specific selection test applied to the four KEGG GAG pathways should be corrected for multiple comparisons. **Response:** we FDR-corrected across the four pathways; three remain significant (FDR = 0.01). This is an addendum to the selection scan in this folder.

In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np
import pybedtools
from pybedtools import BedTool
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools
import re
import ast
import math

# --- Shared helpers: const.py lives in analyses/common ---
NB_DIR = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(NB_DIR, '..', 'common')))
import const
from const import pos_active_ctrl_color, neg_active_ctrl_color, highlight_color, custom_cmap
from const import set_equal_plot_limits, plot_color_pallete, custom_cmap_bolder, FONT_SIZE_small
const.set_plot_style()

# --- Output directory (EDIT): where plots/tables are written ---
output_path = '.'


# Adjusting the P-values of the selection test, to account for the multiple testing of GAG-related pathways. 

In [ ]:
from statsmodels.stats.multitest import multipletests
import pandas as pd

# Original P values
p_values = [0.007451985, 0.007011986, 0.258955482, 0.004413991]
pathway_ids = ["hsa00532", "hsa00534", "hsa00533", "hsa00531"]
pathway_names = [
    "Glycosaminoglycan biosynthesis - chondroitin sulfate / dermatan sulfate",
    "Glycosaminoglycan biosynthesis - heparan sulfate / heparin",
    "Glycosaminoglycan biosynthesis - keratan sulfate",
    "Glycosaminoglycan degradation"
]

# Apply FDR correction (Benjamini-Hochberg method)
reject, adj_pvalues, _, _ = multipletests(p_values, method='fdr_bh')

# Create results dataframe
results = pd.DataFrame({
    'Pathway ID': pathway_ids,
    'Pathway Name': pathway_names,
    'Original P-value': p_values,
    'FDR-adjusted P-value': adj_pvalues,
    'Significant (α=0.05)': reject
})

print("FDR-Adjusted P-values (Benjamini-Hochberg method):")
print(results.to_string(index=False))


In [ ]:
df

In [ ]:
# Load the dataset
dataset_path = "/home/labs/davidgo/Collaboration/humanMPRA/paper/extended_datasets/analysis_results_idan_selection_test.csv"
df = pd.read_csv(dataset_path)

# Filter for the four pathways (with path: prefix)
pathways_to_filter = ["path:hsa00532", "path:hsa00534", "path:hsa00533", "path:hsa00531"]
df_filtered = df[df['pathway ID'].isin(pathways_to_filter)].copy()

# Calculate FDR only on these four pathways
from statsmodels.stats.multitest import multipletests
reject, fdr_values, _, _ = multipletests(df_filtered['p_value'], method='fdr_bh')

# Add FDR as a new column
df_filtered['FDR'] = fdr_values

df_filtered = df_filtered[['pathway ID','L', 'p_value', 'FDR']]

print("Filtered dataset with FDR (calculated on these 4 pathways only):")
print(df_filtered)

In [ ]:
# Save the filtered dataframe with FDR to CSV
output_csv_path = '/home/labs/davidgo/Collaboration/humanMPRA/paper/extended_datasets/selection_test_filtered_pathways_with_fdr.csv'
df_filtered.to_csv(output_csv_path, index=False)
print(f"Dataset saved to: {output_csv_path}")